In [7]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.dates as dates
from datetime import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import pygwalker as pyg
import squarify

### Matplotlib 한글 폰트 설정

In [1]:
# 시스템 명령어로 폰트 설치 (Docker/Jupyter 환경)
!apt-get update -qq
!apt-get install -y fonts-nanum -qq

# matplotlib 버전 확인
import matplotlib as mpl
print(f"Matplotlib 버전: {mpl.__version__}")

# 버전에 따른 폰트 캐시 재구성
try:
    # 최신 버전
    from matplotlib import font_manager
    font_manager.fontManager.addfont('../../fonts/NanumGothic.ttf')
except:
    try:
        # 이전 버전
        mpl.font_manager._rebuild()
    except:
        # 가장 이전 버전
        import matplotlib.font_manager as fm
        fm._rebuild()
        
# 설치된 폰트 확인
import matplotlib.font_manager as fm
fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
print("설치된 나눔 폰트:", fonts)

Matplotlib 버전: 3.10.1
설치된 나눔 폰트: ['NanumMyeongjo', 'NanumBarunGothic', 'NanumMyeongjo', 'NanumSquareRound', 'NanumBarunGothic', 'NanumGothic', 'NanumGothic', 'NanumSquare', 'NanumSquare', 'NanumSquareRound', 'NanumGothic']


In [4]:
# 나눔고딕 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지


#### 데이터 구성 확인 및 중복 데이터 확인

In [8]:
df = pd.read_parquet('../data/train.parquet')

In [9]:
# {데이터 크기 확인}
print(f"데이터 크기: {df.shape}")

데이터 크기: (8350311, 8)


In [10]:
# {데이터 컬럼 확인}
print(f"데이터 컬럼:")
print(df.columns.tolist())

데이터 컬럼:
['user_id', 'item_id', 'user_session', 'event_time', 'category_code', 'brand', 'price', 'event_type']


In [11]:
# {데이터 타입 확인}
print(f"데이터 타입:")
print(df.dtypes)

데이터 타입:
user_id           object
item_id           object
user_session      object
event_time        object
category_code     object
brand             object
price            float64
event_type        object
dtype: object


In [12]:
# {데이터 결측치 확인}
print(f"결측치 확인:")
print(df.isnull().sum())

결측치 확인:
user_id          0
item_id          0
user_session     0
event_time       0
category_code    0
brand            0
price            0
event_type       0
dtype: int64


In [13]:
# {중복 데이터 확인(모든 컬럼의 값이 동일한 행)}
duplicate_count = df.duplicated().sum()
print(f"중복 데이터 수: {duplicate_count}")

중복 데이터 수: 18


In [23]:
# delete duplicates(중복 제거 후 df)
df_dedup = df.drop_duplicates()
print(f"\n중복 제거 전 데이터 크기: {df.shape}")
print(f"\n중복 제거 후 데이터 크기: {df_dedup.shape}")
print(f"\n제거된 행 수: {df.shape[0] - df_dedup.shape[0]}")

# 중복 제거된 df으로 업데이트
df = df_dedup


중복 제거 전 데이터 크기: (8350311, 8)

중복 제거 후 데이터 크기: (8350293, 8)

제거된 행 수: 18


#### event_time 컬럼 값 타입 변경

In [24]:
df.head()

,user_id,item_id,user_session,event_time,category_code,brand,price,event_type
0,0b517454-e7c3-44ec-8c39-a68ef9c0ec60,18c11cbb-a18d-4a9e-bdea-6abd3f7d3c04,ad97f19a-f5fb-41ea-a7b2-52c21fb37ab2,2019-11-16 16:31:26 UTC,apparel.shoes,kapika,72.05,view
1,215eeee5-f9c5-4213-8641-7561dbdad1b9,47c5a6da-32d0-4a29-8b51-57304f476ded,6058b45b-bdb9-4d6c-b300-42dcb1cb8280,2019-11-04 18:59:50 UTC,apparel.shoes,respect,82.63,view
2,a25bf14a-49ac-49bb-87de-ee6b300f0cc4,a6d915c6-2bb7-4393-a556-c327723d3666,28a8b8e3-b374-435d-9d5d-b96058ecb75b,2019-11-26 09:01:47 UTC,apparel.tshirt,goodloot,24.43,view
3,09ee8591-25e0-4bb4-ae24-c48ed4212e3c,0fd4da5d-989c-4a75-9ace-2b108f834c8c,f2972db7-9916-4a58-b6f9-c76afde6245e,2019-11-15 16:05:34 UTC,apparel.shoes,baden,70.79,view
4,7acf7c81-69f6-4aa8-b19f-8e85aeaffc28,d52d1c91-5534-4de4-aaf1-318e932e10e7,7d46d970-b40e-4a2f-81a7-65bf23aa0aae,2019-11-16 13:14:09 UTC,apparel.shoes,rooman,53.80,view


In [25]:
# event_time 컬럼 포맷 변경
df['event_time'] = pd.to_datetime(df['event_time'], format='%Y-%m-%d %H:%M:%S %Z')
df = df.sort_values(by='event_time', ascending=True)

In [26]:
df.head()

,user_id,item_id,user_session,event_time,category_code,brand,price,event_type
501019,24d3ec59-5019-4edd-9cbc-1b33ae7808a4,e4c8cedc-4107-497f-b134-caf123fbe6a2,aa044ff4-3a74-4fd8-b68b-9b0c9a3fe1e8,2019-11-01 00:00:17+00:00,apparel.tshirt,goodloot,8.73,view
1166610,33210fa3-230c-4b1b-946a-2374d0b210c8,6612cfd9-5e1f-4f34-ad97-cd858c70a15e,09c35385-b085-498c-8828-615f6e7c147b,2019-11-01 00:01:21+00:00,apparel.shirt,jordan,102.71,view
1166611,33210fa3-230c-4b1b-946a-2374d0b210c8,57680301-7d01-4ed7-8d21-6a39ecb6f989,09c35385-b085-498c-8828-615f6e7c147b,2019-11-01 00:01:41+00:00,apparel.shirt,jordan,102.71,view
1166612,33210fa3-230c-4b1b-946a-2374d0b210c8,f0205dd1-ff73-4726-a8ba-13b6ecd34896,09c35385-b085-498c-8828-615f6e7c147b,2019-11-01 00:02:44+00:00,apparel.trousers,jordan,83.53,view
1204577,33210fa3-230c-4b1b-946a-2374d0b210c8,ef48c327-e9a4-415a-9a56-94b3a173fe55,2a5ee8b6-a608-4f09-b558-0e2351cb6bf2,2019-11-01 00:04:05+00:00,apparel.shoes,rooman,48.39,view


In [33]:
user_count = df['user_id'].nunique()
print(f"고유 사용자 수: {user_count}")

고유 사용자 수: 638257


In [34]:
item_count = df['item_id'].nunique()
print(f"고유 아이템 수: {item_count}")


고유 아이템 수: 29502


In [35]:
session_count = df['user_session'].nunique()
print(f"고유 세션 수: {session_count}")

고유 세션 수: 2889552


In [36]:
# 사용자(user), 아이템(item), 섹션(user_session) 인덱스로 매핑하기 위한 딕셔너리 생성
user2idx = {v: k for k, v in enumerate(df['user_id'].unique())}  # 각 사용자를 인덱스로 매핑
item2idx = {v: k for k, v in enumerate(df['item_id'].unique())}  # 각 아이템을 인덱스로 매핑
session2idx = {v: k for k, v in enumerate(df['user_session'].unique())}  # 각 세션을 인덱스로 매핑

# 데이터 프레임에 인덱스 컬럼 추가
df['user_idx'] = df['user_id'].map(user2idx)
df['item_idx'] = df['item_id'].map(item2idx)
df['session_idx'] = df['user_session'].map(session2idx)


In [37]:
df.head()

,user_id,item_id,user_session,event_time,category_code,brand,price,event_type,user_idx,item_idx,session_idx
501019,24d3ec59-5019-4edd-9cbc-1b33ae7808a4,e4c8cedc-4107-497f-b134-caf123fbe6a2,aa044ff4-3a74-4fd8-b68b-9b0c9a3fe1e8,2019-11-01 00:00:17+00:00,apparel.tshirt,goodloot,8.73,view,0,0,0
1166610,33210fa3-230c-4b1b-946a-2374d0b210c8,6612cfd9-5e1f-4f34-ad97-cd858c70a15e,09c35385-b085-498c-8828-615f6e7c147b,2019-11-01 00:01:21+00:00,apparel.shirt,jordan,102.71,view,1,1,1
1166611,33210fa3-230c-4b1b-946a-2374d0b210c8,57680301-7d01-4ed7-8d21-6a39ecb6f989,09c35385-b085-498c-8828-615f6e7c147b,2019-11-01 00:01:41+00:00,apparel.shirt,jordan,102.71,view,1,2,1
1166612,33210fa3-230c-4b1b-946a-2374d0b210c8,f0205dd1-ff73-4726-a8ba-13b6ecd34896,09c35385-b085-498c-8828-615f6e7c147b,2019-11-01 00:02:44+00:00,apparel.trousers,jordan,83.53,view,1,3,1
1204577,33210fa3-230c-4b1b-946a-2374d0b210c8,ef48c327-e9a4-415a-9a56-94b3a173fe55,2a5ee8b6-a608-4f09-b558-0e2351cb6bf2,2019-11-01 00:04:05+00:00,apparel.shoes,rooman,48.39,view,1,4,2


In [38]:
# {데이터 타입 다시 확인} : 
print(f"데이터 타입:")
print(df.dtypes)

데이터 타입:
user_id                       object
item_id                       object
user_session                  object
event_time       datetime64[ns, UTC]
category_code                 object
brand                         object
price                        float64
event_type                    object
user_idx                       int64
item_idx                       int64
session_idx                    int64
dtype: object
